In [ ]:
# Lambda Labs Jupyter Optimization for YOLO Hyperparameter Tuning
import os
import sys
import yaml
import platform
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset
from datetime import datetime
from typing import Dict, Any, Optional, Tuple
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Lambda Labs specific: Set matplotlib backend for headless environment
plt.switch_backend('Agg')  # Prevents GUI issues
%matplotlib inline

# Display system info
print(f"🖥️ System Info:")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Platform: {platform.platform()}")

In [ ]:
# Lambda Labs specific configurations
class LambdaLabsYOLOTuner:
    def __init__(self, config_path: str = "config.yaml"):
        self.config_path = Path(config_path)
        self.config = self._load_config()
        self.project_root = Path().resolve()
        self.tune_config = self.config['tune']
        self.results_dir = None
        
        # Lambda Labs optimizations
        self._setup_lambda_environment()
        self._verify_dataset()
    
    def _setup_lambda_environment(self):
        """Setup optimized for Lambda Labs environment"""
        # Configure Ultralytics datasets directory
        settings.update({"datasets_dir": str(self.project_root)})
        
        # Lambda Labs: Disable unnecessary GUI components
        os.environ['DISPLAY'] = ''  # Prevent X11 issues
        
        # Force CPU for matplotlib to prevent memory leaks
        import matplotlib
        matplotlib.use('Agg')
        
        print(f"✅ Lambda Labs environment configured")
        print(f"📁 Project root: {self.project_root}")
        print(f"🗃️ Datasets dir: {settings['datasets_dir']}")
    
    def _load_config(self) -> Dict[str, Any]:
        """Load config with error handling for cloud environment"""
        try:
            with open(self.config_path, 'r') as f:
                config = yaml.safe_load(f)
            print(f"✅ Configuration loaded from: {self.config_path}")
            return config
        except Exception as e:
            print(f"❌ Error loading config: {e}")
            # Lambda Labs: Show current directory contents for debugging
            print("📁 Current directory contents:")
            for item in Path().iterdir():
                print(f"  {item.name}")
            raise

In [ ]:
def _verify_dataset(self):
    """Verify dataset with Lambda Labs path handling"""
    dataset_yaml_path = Path(self.config["dataset_yaml_path"]).resolve()
    
    print(f"🔍 Looking for dataset: {dataset_yaml_path}")
    
    if not dataset_yaml_path.exists():
        # Lambda Labs: Show alternative paths that might exist
        print("❌ Dataset YAML not found. Checking alternatives...")
        possible_paths = [
            Path("data/powerlines/original/original.yaml"),
            Path("./data/powerlines/original/original.yaml"),
            Path("../data/powerlines/original/original.yaml")
        ]
        for path in possible_paths:
            if path.exists():
                print(f"✅ Found alternative: {path}")
                # Update config
                self.config["dataset_yaml_path"] = str(path)
                dataset_yaml_path = path
                break
        else:
            raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml_path}")
    
    # Clean and verify dataset
    try:
        data = yaml.safe_load(dataset_yaml_path.read_text())
        modified = False
        
        if data.pop("path", None) is not None:
            print("🔄 Removed stale 'path' key from dataset YAML")
            modified = True
        
        if data.get("nc") != 1 or data.get("names") != ["powerline"]:
            data["nc"] = 1
            data["names"] = ["powerline"]
            modified = True
            print("🔄 Updated dataset to single class 'powerline'")
        
        if modified:
            dataset_yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
            print(f"✅ Dataset YAML cleaned: {dataset_yaml_path}")
        
        check_det_dataset(str(dataset_yaml_path))
        print("✅ Dataset structure verified")
        
    except Exception as e:
        print(f"⚠️ Dataset verification warning: {e}")

# Add method to class
LambdaLabsYOLOTuner._verify_dataset = _verify_dataset

In [ ]:
def _prepare_search_space(self) -> Dict[str, Tuple[float, float]]:
    """Prepare search space for Lambda Labs"""
    search_space = {}
    
    for param, value_range in self.tune_config['search_space'].items():
        if isinstance(value_range, list) and len(value_range) == 2:
            search_space[param] = tuple(value_range)
        else:
            print(f"⚠️ Skipping parameter {param}: invalid range format {value_range}")
    
    print(f"🎛️ Search space prepared with {len(search_space)} parameters")
    return search_space

def _initialize_model(self) -> YOLO:
    """Initialize model with Lambda Labs GPU optimization"""
    model_path = self.config['model_type']
    print(f"🚀 Initializing model: {model_path}")
    
    try:
        # Lambda Labs: Ensure model downloads to local storage
        model = YOLO(model_path)
        
        # Move to GPU if available
        if torch.cuda.is_available():
            print(f"🎮 Model moved to GPU: {torch.cuda.get_device_name(0)}")
        
        print(f"✅ Model loaded successfully")
        return model
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        raise

# Add methods to class
LambdaLabsYOLOTuner._prepare_search_space = _prepare_search_space
LambdaLabsYOLOTuner._initialize_model = _initialize_model

In [ ]:
def run_tuning_lambda(self) -> Optional[Any]:
    """Execute tuning optimized for Lambda Labs environment"""
    search_space = self._prepare_search_space()
    model = self._initialize_model()
    
    dataset_yaml_path = Path(self.config["dataset_yaml_path"]).resolve()
    
    # Lambda Labs optimized arguments
    tuning_args = {
        'data': str(dataset_yaml_path),
        'epochs': self.tune_config['epochs'],
        'iterations': self.tune_config['iterations'],
        'optimizer': self.tune_config['optimizer'],
        'plots': False,  # Disable plots to save memory/time on Lambda Labs
        'save': False,   # Only save best results to save storage
        'val': False,    # Skip intermediate validation for speed
        'device': self.config['device'],
        'patience': self.tune_config['patience'],
        'space': search_space,
        'project': str(self.project_root / 'results' / 'tune'),  # Use absolute path
        'name': f"lambda_tune_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        'exist_ok': True,
        'verbose': True  # Keep verbose for Jupyter output
    }
    
    if self.tune_config.get('resume', False):
        tuning_args['resume'] = True
    
    # Display configuration
    print("🎯 Lambda Labs Tuning Configuration:")
    for key, value in tuning_args.items():
        if key != 'space':
            print(f"  {key}: {value}")
    
    # Time estimation
    estimated_hours = (self.tune_config['epochs'] * self.tune_config['iterations']) / 60
    print(f"\n⏱️ Estimated time: {estimated_hours:.1f} hours")
    print(f"💰 Estimated cost: ~${estimated_hours * 1.29:.2f} (A100 40GB)")
    
    # Confirm before starting
    response = input("\n🤔 Continue with tuning? (y/N): ")
    if response.lower() != 'y':
        print("❌ Tuning cancelled")
        return None
    
    start_time = datetime.now()
    print(f"🕐 Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    try:
        # Clear output periodically to prevent notebook bloat
        results = model.tune(**tuning_args)
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print(f"\n✅ Hyperparameter tuning completed!")
        print(f"🕐 End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⏱️ Total duration: {duration}")
        
        if hasattr(results, 'save_dir'):
            self.results_dir = Path(results.save_dir)
            print(f"📁 Results saved to: {self.results_dir}")
        
        return results
        
    except Exception as e:
        print(f"❌ Error during tuning: {e}")
        # Lambda Labs: Save partial results if possible
        print("💾 Attempting to save partial results...")
        return None

# Add method to class
LambdaLabsYOLOTuner.run_tuning_lambda = run_tuning_lambda

In [ ]:
def analyze_results_lambda(self):
    """Analyze results with Lambda Labs optimizations"""
    if not self.results_dir:
        # Try to find results directory
        results_base = self.project_root / 'results' / 'tune'
        if results_base.exists():
            tune_dirs = list(results_base.glob('*/tune'))
            if tune_dirs:
                self.results_dir = max(tune_dirs, key=lambda x: x.stat().st_mtime)
                print(f"📊 Found results directory: {self.results_dir}")
    
    if not self.results_dir or not self.results_dir.exists():
        print("⚠️ No results directory found")
        return None
    
    # Load results CSV
    results_csv = self.results_dir / 'tune_results.csv'
    if not results_csv.exists():
        print("⚠️ tune_results.csv not found")
        return None
    
    try:
        results_df = pd.read_csv(results_csv)
        print(f"📈 Loaded {len(results_df)} tuning iterations")
        
        # Display summary statistics
        print("\n📊 Results Summary:")
        print(f"Best fitness: {results_df['fitness'].max():.4f}")
        print(f"Mean fitness: {results_df['fitness'].mean():.4f}")
        print(f"Std fitness: {results_df['fitness'].std():.4f}")
        
        # Show best parameters
        best_idx = results_df['fitness'].idxmax()
        best_result = results_df.iloc[best_idx]
        
        print(f"\n🏆 Best iteration #{best_idx}:")
        print(f"Fitness: {best_result['fitness']:.4f}")
        
        # Create simple visualization (Lambda Labs friendly)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Fitness progression
        ax1.plot(results_df.index, results_df['fitness'])
        ax1.axhline(y=results_df['fitness'].max(), color='red', linestyle='--', label='Best')
        ax1.set_title('Fitness Progression')
        ax1.set_xlabel('Iteration')
        ax1.set_ylabel('Fitness')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Fitness distribution
        ax2.hist(results_df['fitness'], bins=20, alpha=0.7, edgecolor='black')
        ax2.axvline(x=results_df['fitness'].max(), color='red', linestyle='--', label='Best')
        ax2.set_title('Fitness Distribution')
        ax2.set_xlabel('Fitness')
        ax2.set_ylabel('Count')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save plot
        plot_path = self.results_dir / 'lambda_analysis.png'
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        print(f"📊 Plot saved: {plot_path}")
        
        plt.show()
        
        return results_df
        
    except Exception as e:
        print(f"❌ Error analyzing results: {e}")
        return None

# Add method to class
LambdaLabsYOLOTuner.analyze_results_lambda = analyze_results_lambda

In [ ]:
# Initialize and run tuning
print("🚀 Starting Lambda Labs YOLO Hyperparameter Tuning")
print("=" * 60)

# Initialize tuner
tuner = LambdaLabsYOLOTuner('config.yaml')

# Run tuning
results = tuner.run_tuning_lambda()

# Analyze results
if results:
    print("\n📊 Analyzing results...")
    results_df = tuner.analyze_results_lambda()
    
    if results_df is not None:
        print(f"\n🎉 Tuning completed successfully!")
        print(f"Best fitness achieved: {results_df['fitness'].max():.4f}")
    else:
        print("⚠️ Results analysis failed")
else:
    print("❌ Tuning failed or was cancelled")

In [ ]:
# Extract and display best hyperparameters
if tuner.results_dir:
    best_params_file = tuner.results_dir / 'best_hyperparameters.yaml'
    
    if best_params_file.exists():
        print("🏆 BEST HYPERPARAMETERS:")
        print("=" * 50)
        
        with open(best_params_file, 'r') as f:
            content = f.read()
            print(content)
        
        print("\n📋 Next Steps:")
        print("1. Copy the hyperparameters above")
        print("2. Update your main config.yaml")
        print("3. Run full training with optimized parameters")
        print("4. Download results before terminating Lambda Labs instance")
    else:
        print("⚠️ Best hyperparameters file not found")
else:
    print("⚠️ No results directory available")